# Welcome!

Hello James, welcome to this tutorial on some of the functionalities of the code that I used for analysis of gravitational waves in white dwarf-compact object mergers! Eventually, your goal will be to modify this code for use in determining gravitational waveforms for neutron stars embedded in dark matter halos. Here, I try to walk through some of the calculations necessary for my problem (which you can read about in Section 2 of my paper), but I also recommend you to try things out on your own to see how things work and how they break. First, let's install the packages that my code needs, and then the code itself.

In [ ]:
using Pkg          
Pkg.activate(".")
#The above lines will tell the project the full list of packages that are needed for my project, 
#and the packages that those packages need, and the packages that those packages need, etc.
#These can be found in .project/.toml files. Unless strictly necessary, I would not recommend editing these.
#The computer will do that for you

#And these are the packages that I want to use directly at some point.
using BenchmarkTools
using Integrate
using CSV, DataFrames, Interpolations
using Plots
using OrdinaryDiffEq
using DifferentialEquations
using Printf, DelimitedFiles
using ForwardDiff
using LaTeXStrings

The above are external dependencies, and below are my modules. We will see that to use them, the code will be `module.function()`. 

In [ ]:
include("src/Integrate.jl")
include("src/Math.jl")
include("src/Integrate_r.jl")
include("src/Save.jl")
include("src/Pipeline.jl")

# Julia

I use a programming language called Julia for my code development. Julia is syntactiacally similar to Python in that they are both high-level languages, but under the hood it is what is called a "Just in Time" (JIT) compiler. This means that despite being close to human language, it is fast like lower-level languages such as C, C++, or Fortran. Starting with the mandatory `Hello, world`, let's see some of the basics of Julia.

In [ ]:
println("Hello, world!")

In [ ]:
arr=[1,2,3,4,5]
println(arr[1])
println(arr[end])
println(arr[2:4])
println(arr[1:2:end])
println(arr[arr.%2 .==0])#What does this mean?

In [ ]:
println(arr[0])#Try running this like you would in Python.

In [ ]:
push!(arr,6)#What is the difference between this...

arr2=[7,8]


append!(arr,arr2)#this,

arr3=[9,10]

arr4=vcat(arr, arr3)#and this? 
#Try printing different values at different points along this sequence to figure it out!
#Bonus: What do you think the exclamation point means?

If you have experience in Python, you may notice that indexing and slicing work a bit differently than you were expecting. How do they differ? Try playing around with the following slice, including calling numbers from it and (to Tommu Tutone's chagrin) mutating it: 

In [ ]:
jenny=[8,6,7,5,3,0,9]

In [ ]:
#jenny, don't change your number
#just kidding, change that thing!



Let's try some loops and functions now. What do you think will happen when you run each of these blocks?

In [ ]:
sum=0

for i in 1:100
   sum+=i
end

println(sum)

In [ ]:

for i in range(0, stop=1, length=11)
   println(i)
end


In [ ]:
arr_append=[]
for i in 1:100
   if i%10==3 #What does "%" mean?
        push!(arr_append,i)
    end
end
println(arr_append)

In [ ]:
square=0
index=0

while square<200
    square+= index
    index+=1
    square+=index
    println(square)
end

In [ ]:
for i in 1:100
    if sqrt(i)%1==0 || cbrt(i)%1==0 #what does "\\" mean?
        println(i)
    end
end

In [ ]:
#try changing the above loop so that it will only print numbers that are both squares and cubes between 1 and 100 (Hint: it should print 1 and 64)

Notice the `end` structure used to close loops in lieu of Pythons indentation (the indendation here is just to make it easier for us to read. For instance:

In [ ]:
snarky=["Look ", "Ma, ",  "no ", "indents!"]
for i in 1:8
if i%2==0
print(snarky[round(Int,i/2)])
end
end

Okay, now for some functions:

In [ ]:
function addition(m, n)
    return(m+n)
end

println(addition(60, 7))

In [ ]:
function circumference(r)
    return(2*π*r)#that's right, you can use unicode symbols in Julia!
end

println(circumference(5))

In [ ]:
function quadratic_formula(a,b,c)
    return( (-b+ sqrt(b^2-4*a*c) )/(2*a) )
end

println(quadratic_formula(1,-2,1))

In [ ]:
println(quadratic_formula(1,0,1))

You notice that we had a problem with `quadratic_formula` when we tried to take the square root of a negative number. (Whoops). Perhaps we can fix this by telling the function to expect to deal with complex numbers: 

In [ ]:
function quadratic_formula(a::Complex{Int64},b::Complex{Int64},c::Complex{Int64})::Complex{Int64}
    return( (-b+ (b^2-4*a*c)^(0.5) )/(2*a) )
end

println(quadratic_formula(1+0im,0+0im,1+0im))

In the above, I told julia that `a`, `b`, and `c` are complex integers, and that the output is also a complex integer. While these functions are simple, with more complicated functions or with functions called many times, this typecasting can help to reduce memory and speed up functions by allowing Julia to pre-allocate memory. Let's try it with our other functions, too: 

In [ ]:
function addition(m::float, n::float)::float
    return(m+n)
end

function circumference(r::float)::int
    return(2*π*r)
end

println(addition(60, 7))
println(circumference(5))

In [ ]:
π

What went wrong with the above functions? See if you can fix it.

Much like `numpy` arrays in Python, Julia arrays can be acted upon pointwise. Here, a `.` must be used to tell Julia to apply the operation pointwise. For example:

In [ ]:
[1,2,3].+[1,2,3]

In [ ]:
[2,-1,0,1,2].*[0.01,0.1,1,10,100]

What is wrong with the following code (Hint: Run the code and read the error statement)?

In [ ]:
twos = 2.*ones(10)

twos .^ range(1,stop=10, length=10)

See if you can make the following function work (Hint: It is only one character away from working): 

In [ ]:
addition([1,2,3], 1)

As in numpy arrays, you can make multidimensional matrices in Julia:

In [ ]:
lst2D=[[11,12,13],
[21,22,23],
[31,32,33]]

println(lst2D[1][1], " ", lst2D[2][1], " ",lst2D[3][2])

`lst2D` is not quite what we want, though, as Julia parses it more as a "vector of vectors" than as a true 2D matrix. Before proceding to the next blocks, try playing around with `lst2D` (perhaps in conjunction with some of the functions above) to explore these limitations.

In [ ]:
#play around with `lst2D` here...

In [ ]:
#and here...

In [ ]:
#and here.

Now we can make `lst2D` into a true 2D matrix and see some of the nice features that this affords us:

In [ ]:
arr2D=[
    11 12 13;
    21 22 23;
    31 32 33;
]

println(arr2D)
#equivalently, arr2D = reduce(vcat, permutedims.(lst2D))

In [ ]:
println(arr2D[1,1], " ", arr2D[2,1], " ",arr2D[3,2])#We can now call 2D indices more easily

In [ ]:
arr2D.+1 #we can do pointwise operations with scalars...

In [ ]:
another_arr=[
    1 1 1;
    2 2 2;
    3 3 3
]

arr2D.*another_arr #or other matrices

In [ ]:
transposed_arr=arr2D'#we can transpose the matrix (i.e., flip it about its diagonal)

In [ ]:
#We can concatenate matrices row-wise or column-wise:

three_arr=3 .* ones(3,3)

plus_three=arr2D.+three_arr

plus_30= @.arr2D+10*three_arr 
#I have done something a bit trickier here--I have used a macro (@) to apply the pointwise `.` to all operations
#i.e., this is equivalent to plus_30= arr2D.+10.*three_arr

horizontal_arr=hcat(arr2D, plus_three)#equivalently [arr2D plus_three]

In [ ]:
vertical_arr=vcat(arr2D, plus_30)#equivalently [arr2D; plus_30]

In [ ]:
#We can access individual colums or rows:

vertical_arr[:,2]

In [ ]:
println(vertical_arr[5])#What does this do? Is it different than Python?

#println(vertical_arr[10])#What do you think this will print?

#vertical_arr[5,:]

If we want to get really tricky, we can even try adding a vector pointwise to each row:

In [ ]:
arr_100=[100, 200, 300]

#What do you think the following two will print? Take a guess before running the code
#println(arr2D.+arr_100)
#println(arr2D.+arr_100')

Given the results of the last two cells, can we say anything about how Julia treats rows vs. columns? Is this the same or different from how Python treats rows vs. columns? (These might be called "row/column-major" languages. How would you classify Julia/Python as row/column-major?)

It can also prove useful to visualize your data to determine whether it is reasonable or to find general trends. I strongly prefer using Python's Matplotlob package to make paper-quality plots, but I have found it necessary to have a working (if simple) understanding of Julia's `Plots.jl` to check my work. As such, the following is a very basic rundown of how to use `Plots.jl` and its most basic functionalities, but there are many more in-depth tutorials online ([for instance](https://www.youtube.com/watch?v=3aj3D2phXFA)).  

In [ ]:
x=[0,1,2,3]
y=x.^2

plot(x,y)#It is very easy to get started with plots

In [ ]:
#It is always good practice to label x and y coordinates, and Plots.jl includes a legend for you automatically:
plot(x,y, xlabel = "x", ylabel = "y", label="Practice")

In [ ]:
#Suppose that you want to plot two lines at the same time. 
#You can use the "!" macro to plot "in-place". 
#Run the following code to see what happens, then try to plot the two lines at on the same axis.

x2=range(0,stop=3,length=101)
y2=x2.^2

plot(x,y, xlabel = "x", ylabel = "y", label="Practice")
plot!(x2, y2, label="Better Practice")

In [ ]:
#You can also make a scatter plot for discrete data:
scatter(x,y, xlabel = "x", ylabel = "y", label="Discrete")
plot!(x2, y2, label="Continuous")

Okay! Now it's your turn. Try using what you have learned in Julia so far to make the below three plots:
1. Make a scatterplot where the x-axis is $N$ from 1 to 100, and the y-axis is the $N^{\rm th}$ prime number.
2. For $n=1\dots5$, plot $y_n=\sum_{j=0}^n(-1)^j\frac{x^{2j}}{(2j)!}$ for $x\in[-2\pi,+1\pi]$ on the same plot with appropriate labels. (If you feel so compelled, try plotting the residual $y_n-cos(x)$.) 
3. (Challenge) For a grid of $c=x+iy$, let $z_n(c)=z_{n-1}^2+c$ with $z_0(c)\equiv0$. For a grid of $x\in[-2,1],\,y\in[-1,+1]$, calculate $z_{10}(c)$, and color the grid where $|z_{10}(c)|<2$. 

Now, I will get started on the specifics of my code. There are many other promers on Julia in case you are interested, including:
1. [Julia in 100 seconds](https://www.youtube.com/watch?v=JYs_94znYy0)
2. [Julia Academy: Introduction to Julia](https://github.com/JuliaAcademy/Introduction-to-Julia)
3. [Think Julia](https://benlauwens.github.io/ThinkJulia.jl/latest/book.html)
4. [Computational Thinking](https://computationalthinking.mit.edu/Fall23)

You can reference these as desired, but do not feel that you need to read them right now. As opposed to reading a textbook, I have often found that the best way to learn a coding language is by trying, crashing and burning, and then asking Stack Exchange (or I guess ChatGPT nowadays) for help. With that said, let's take a look at some white dwarfs!

# Math.jl

Like many problems in physics, what we are trying to do here boils down to solving a differential equation. In Section 2 of my paper, I lay out the set of equations that we need to solve. In our case, we have five key parameters that we need to solve for: White dwarf/compact binary mass ($M_{\rm WD},\,M_{\rm C}$), white dwarf radius ($R_{\rm WD}$), orbital phase ($\theta$), and binary separation ($a$). We assume a circular binary, so we don't need to worry about eccentricity.

NOTE: Units are crucial in any problem of physics! For the evolution of the binary, I use highly unconventional units of km-$\M_\odot$-s, which is to say that any unis of length are in kilometers, any units of mass are in solar masses, any unit of time is in seconds, and compound units are implied by these three. (For comparison, astronomers usually use centimeter-gram-second (cgs) units.) In your code, you will likely be using natural/geometrized units with $G=c=1$, but we can discuss that later.

Let's start with orbital phase. Given Kepler's Third Law ($\left(\frac{T}{2\pi}\right)^2=\frac{a^3}{GM_{\rm tot}}$), can you derive how phase should change with time? (i.e., $\theta=2\pi\frac{t}{T}$, find $\dot\theta=\frac{\partial \theta}{\partial t}$) Try coding a function below for $\dot\theta$. In this case, 

In [ ]:
"""Hint: It may be useful to write a helper function to determine period:
function period(a,M_tot)
    #your code here
end
"""


function dot_theta(a,M_WD,M_C)
    #your code here
end

The way that you determine $\dot \theta$ using my code is `Math.dphase(0,a,M_WD, M_NS)` (don't worry about the first parameter--it represents angular momentum used for a different method of calculation during which the orbital period is changing very quickly, while we assume that $\dot T\approx 0$). Try comparing your method to mine to see if we get the same answer:

In [ ]:
as=range(100.,stop=30000.,length=300)
M_Cs=10 .^range(-2,stop=2,length=201)

dt_James_as=dot_theta.(as,1,1.4)
dt_Tristan_as=Math.dphase.(0,as,1., 1.4)

dt_James_Ms=dot_theta.(10000,1,M_Cs)
dt_Tristan_Ms=Math.dphase.(0,10000.,1., M_Cs)

plot(as,dt_James_as,label="James")
plot!(as,dt_Tristan_as,label="Tristan")
plot!(xlabel="a [km]",ylabel=L"\dot\theta\ [\mathrm{ rad}\ \mathrm{s}^{-1}]", xscale=:log10, yscale=:log10)

In [ ]:
plot(M_Cs,dt_James_Ms,label="James")
plot!(M_Cs,dt_Tristan_Ms,label="Tristan")
plot!(xlabel=L"\mathrm{M}_\mathrm{C}\ [M_\odot]",ylabel=L"\dot\theta\ [\mathrm{ rad}\ \mathrm{s}^{-1}]",
    xscale=:log10, yscale=:log10
)

To determine how quickly matter is flowing off of the WD, we must know how much of the WD sits outside of its gravitational well of influence, so to speak. To know this, we must know how large the WD is. It has been found that the radius of a WD *decreases* as the WD's mass increases. In particular, not accounting for relativistic effects, $R_{\rm WD}=7792\,{\rm km}\left(\frac{M_{\rm WD}}{0.7\,M_\odot}\right)^{-1/3}$. Try coding this below, then compare it to my code by calling `Math.R0_WD(M_WD)`.

In [ ]:
function R_WD_non_rel 
        #code here
end

#Now try various white dwarf masses for your function and mine.
#Where are they similar? Where do they differ?
#Does anything weird hapen at certain values of `Math.R0_WD(M_WD)`?
#If so, does that mean my code is broken, or can you think of any physical reason for this?

Now, we can use this radius to determine how quickly mass leaves the WD and how much of it accretes onto the compact object partner. Use `Math.decretion_rate(a, R_WD, M_WD, M_C)` where you calculate the appropriate value for `R_WD` from `M_WD` to determine the "decretion rate" in units of $M_\odot\,{\rm s}^{-1}$ (note what a crazy unit this is!).

In [ ]:
#Hint: You have three different variables, try to see how each of them affects Math.decretion_rate and plot the results.
#Try changing `a` here.

In [ ]:
#And try changing M_WD here.

In [ ]:
#Finally, try changing M_C here.

You can use `Math.dM_NS(a, R_WD, M_WD, M_C)` to compute the accretion rate of the compact object partner. 

Compare the accretion rate with the above decretion rates that you calculate. 



In [ ]:
#calculate accretion rates for the same scenarios as you found above. 

In [ ]:
#compare with decretion rate here. 

In particular, there is an approximate maximum accretion rate (the Eddington accretion rate) after which the accreting material begins glowing so brightly that it pushes away any further accreting material that would like to fall on the compact object (partial explanation [here](https://www.youtube.com/watch?v=cdrxdTMNMf8)). Can you determine this Eddinton accretion rate as a function of `M_C` based on your code output?

In [ ]:
#Try to determine Eddington accretion rate here.

Finally, we must determine the evolution of binary separation. This part gets a bit messier, so bear with me. There are two effects at play here: mass transfer, and radiation reaction. We will start with the effects of mass transfer.

We can derive the effect of mass transfer on separation by viewing the system through the lens of angular momentum. The angular momentum, $J$, of a circular binary with separation $a$ and masses $M_{\rm WD}$ and $M_{\rm C}$ is $$J_{\rm orb}=\sqrt{\frac{GM_{\rm WD}^2M_{\rm C}^2}{M_{\rm WD}+M_{\rm C}}a} $$

From this formula, see if you can calculate $\frac{\dot J}{t}=\frac{\partial \ln J}{\partial t}$ in terms of the masses, the separation, and their derivatives (i.e., Equation 8 in my paper).




Furthermore, we can assume that the angular momentum lost from the system due to mass transfer is equal to: $$ \frac{\dot J}{J} =\gamma (1-\beta)\frac{\dot M_{\rm WD}}{M_{\rm WD} + M_{\rm C}}$$
In this formula, $\beta=\left|\frac{\dot M_C}{M_{\rm WD}}\right|$, and so $(1-\beta)$ represents the fraction of mass lost from the system and $(1-\beta)\frac{\dot M_{\rm WD}}{M_{\rm WD} + M_{\rm C}}=\frac{\partial }{\partial t}(\ln {M_{\rm WD}+M_{\rm C}})$ (i.e., the fractional mass loss rate of the system as a whole). 

$\gamma$ is a parameter describing "fractional specific angular momentum". "Specific angular momentum" is the angular momentum per unit mass, while "fractional" relates this quantity to the angular momentum of the system. (i.e., $\gamma$ is the angular momentum per kilogram of the mass lost divided by the angular momentum per kilogram of the binary)   

See if you can set this equal to the equation that you derived in the last part to recover equation 10 of my paper.

`Math.a_dot(a, R_WD, M_WD, M_NS)` will calculate the derivative of separation under the assumption that $\gamma=\frac{M_{\rm WD}}{M_{\rm C}}$. Try playing around with various values of these parameters to see what you get out. What is the physical interpretation of your results. See if you are able to get both positive and negative outputs for `Math.a_dot(...)`. 

In [ ]:
#call Math.a_dot(...) for several values here and trying to determine the effects of each parameter
#Try making some plots!
#(Hint: Use R_WD=R0_WD(M_WD) as you probably did above.)

In [ ]:
#can you find values for which Math.a_dot(...)>0? What about Math.a_dot(...)<0?

A key challenge in our code is the choice of $\gamma$. Under a different physical assumption, we may instead have $\gamma=\frac{M_{\rm WD}}{M_{\rm C}}$:

In [ ]:
jeans_tuple = (; Math.optional_default..., mode=:jeans) 
println(jeans_tuple.mode)

The above block of code changes the assumption on $\gamma$; by running `Math.a_dot(a, R_WD, M_WD, M_NS; optional=jeans_tuple)`, you use this new assumption. Try using this new assumption and see how it changes things compared to your previous values.  

In [ ]:
#call Math.a_dot(...; optional=jeans_tuple) and compare 
#to values that you got from calling without the optional parameter

In [ ]:
#Is it possible to get both positive and negative values for Math.a_dot(...; optional=jeans_tuple)?
#Compare with your results from Math.a_dot(...) here.

We now move on to the ``binary hardening" of the system due to radiation reaction. This is a fancy way of saying that, as matter accelerates, it produces gravitational waves which carry away energy and angular momentum, meaning that the binary loses energy and angular momentum and subsequently the separation shrinks.

It is substantially more calculated to calculate the effect of radiation reaction because it depends on the derivative of the system parameters (especially since the effect of radiation reaction is an acceleration and radiation reaction depends on the acceleration!). In principle, we could calculate all of the derivatives that we need analytically by hand and repeat this several times to account for the effects of radiation reaction at high order. In practice, this would be rather unpleasant, and I would strongly prefer not to do it.

We are saved by two very clever mathematical inventions: The Post-Newtonian formalism, and `ForwardDiff.jl`. The Post-Newtonian formalism saves us from taking infinite derivatives by expanding the equations of motion due to radiation reaction as a Taylor-like series. `ForwardDiff.jl` takes analytical derivatives for us.

We will quickly discuss derivatives and `ForwardDiff.jl` here. 

In [ ]:
function gaussian(x;μ=0,σ=1)
    return exp(-((x-μ)/σ)^2 /2)/(σ*sqrt(2π))
end

xs_1=range(-1,stop=1,length=201)
#This is likely a familiar function for you, try plotting it on the values of x given
gauss_out_1=#your code here
plot(#your code here)

In [ ]:
#try calculating the derivative of gaussian(x) by hand as an analytical function
function analytic_derivative(x;μ=0,σ=1)
    #your code here 
end

analytic_out=analytic_derivative.(xs_1)

In [ ]:
#Another way that we can take derivatives is numerically--you already have the output, so 
#just use Δy/Δx. We will see that these results are dependent on you well you sample your space

function numerical_derivative(xs,ys)
    #output an array of Δy/Δx 
    #Hint: Your array will likely be one shorter than xs or ys (which should be the same length)
end


#Let's see how well the numerical derivative works:
xs_2=range(-1,stop=1,length=11)
xs_3=range(-1,stop=1,length=31)
xs_4=range(-1,stop=1,length=1001)

ys1=gaussian.(xs_1)
ys2=gaussian.(xs_2)
ys3=gaussian.(xs_3)
ys4=gaussian.(xs_4)

der1=numerical_derivative(xs_1,ys1)
der2=numerical_derivative(xs_2,ys2)
der3=numerical_derivative(xs_3,ys3)
der4=numerical_derivative(xs_4,ys4)

plot(xs_1,analytic_out,label="Analytic Solution")
plot!(xs_2[:end-1],der2,label="Numerical (Very rough)")
plot!(xs_3[:end-1],der3,label="Numerical (Rough)")
plot!(xs_1[:end-1],der1,label="Numerical (Moderate)")
plot!(xs_4[:end-1],der4,label="Numerical (Smooth)")
plot!(xlabel="x",ylabel="f'")

In [ ]:
#Let's zoom in on the residuals:

res1=@.der1-analytic_derivative((xs_1[:end-1]+xs_1[2:end])/2)
res2=@.der2-analytic_derivative((xs_2[:end-1]+xs_2[2:end])/2)
res3=@.der3-analytic_derivative((xs_3[:end-1]+xs_3[2:end])/2)
res4=@.der4-analytic_derivative((xs_4[:end-1]+xs_4[2:end])/2)
#Why did I write the above code like that? Does this make sense?

plot(xs_2[:end-1],res2,label="Very rough")
plot!(xs_3[:end-1],res3,label="Rough")
plot!(xs_1[:end-1],res1,label="Moderate")
plot!(xs_4[:end-1],res4,label="Smooth")
plot!(xlabel="x",ylabel="Derivative Residuals")

In [ ]:
#Alternatively, ForwardDiff.jl will do all of this for us:
dgaussian(x)=ForwardDiff.derivative(gaussian, x)

#Try plotting this derivative and its residuals. 
#How accurate is it compared to the numerical derivatives?

In [ ]:
#Try defining a few more functions and seeing if you can take their derivatives.
#Are there any functions that do not work as well for this?

The function that we will need to take a derivative of is a vector function. Specifically, for $\boldsymbol{\theta}\equiv\{a,\,M_{\rm WD},\,M_{\rm C},\,R_{\rm WD},\,J,\,\phi\}$, we are trying to calculate $\dot{\boldsymbol{\theta}}\equiv\dot{\boldsymbol{\theta}}(\boldsymbol{\theta})$.

In the above example of the gaussian, $f$, I fixed the hyperparameters such that $f\equiv f(x)$. Below, I am going to define the gaussian such that it takes in a vector of arbitrary length $2n+1$ where `x[1:n]` is a spatial vector, `x[n+1:end-1]` is the centroid, and `x[end]=σ`:

In [ ]:
function multivariate_gaussian(x)
    @assert length(x)%2==1 #What does this do? Why did I put it here?
    n=round(Int,(length(x)-1)/2)
    vec=x[1:n]
    μ=x[n+1:end-1]
    σ=x[end]#Note that to be fully general, we could make this a covariance matrix
    #(But I don't feel like doing that.)
    return @. exp(-Base.sum((vec-μ)/σ)^2 /2)/(σ*(2π)^(1/2))^n
end



Now try taking the derivative of `multivariate_gaussian`. You will have to look up how to take a multidimensional derivative with `ForwardDiff.jl`. Good luck!

In [ ]:
#Write a function to take the derivative for arbitrary phase space vector x

In [ ]:
#Test your function and make sure that it works

We need to calculate higher order (up to third) derivatives of the quadrupole moment:
$$I=M_{\rm WD}\vec{x}_{\rm WD}\otimes \vec{x}_{\rm WD} + M_{\rm C}\vec{x}_{\rm C}\otimes \vec{x}_{\rm C} $$
in order to calculate $\dot a_{RR}$. Being slightly clever, we can use `ForwardDiff` repeatedly in conjunction with the chain rule in order to carry out this calculation. (The reason that we cannot naively keep differentiating is that $\dot{\boldsymbol{\theta}}\equiv\dot{\boldsymbol{\theta}}(\boldsymbol{\theta})
\implies
\ddot{\boldsymbol{\theta}}\equiv\ddot{\boldsymbol{\theta}}(\boldsymbol{\theta},\dot{\boldsymbol{\theta}})$.) It took me a month to figure out how to make this part of the code work, so I won't force you to do it here :-) If you want to take a look under the hood, the magic sauce is around lines 341-383 in `Math.jl`.  

To calculate $\ddot I$, you can use `ddI_from_theta([a, M_WD, M_NS, R_WD, J, phase])`. (It does not matter what you use for `J` in this particular calculation.) *Note that you must use the square brackets because derivatives are taken as gradients here.)

In [ ]:
#try calculating the second derivative of I here.
#Do not worry if you get some warning message in red telling you about inefficiencies.  

Okay, we are finally ready to calculate the entire derivative now! Due to the integration package that we will use in the next section, the formatting is a little bit strange. You will want to call `du=zeros(6); acceleration_even!(du, [a, M_WD, M_NS, R_WD, J, phase],[],0)`. 

The `[]` here is for optional parameters which we do not use and the `0` at the end is time, which our system is independent of, so it does not matter what is put there. 

The important quantity here is `du`, which will be used to store the derivative of `theta`$\equiv$`[a, M_WD, M_NS, R_WD, J, phase]` (note that I use `theta` to refer to the full set of parameters). You likely could have guessed that the `acceleration_even!` would store its output in an existing variable--what would have tipped you off about this?

In [ ]:
#Try calculating the full derivative here for several values and interpret the results.

Now that we have all of our integrals set, we need to know where to start our integration. We will start when the WD first overflows its Roche Lobe (sphere of gravitational influence where matter is bound to the star). For $q=\frac{M_{\rm WD}}{M_{\rm C}}$, the Roche Lobe has radius $R_{\rm L}=a\frac{0.49 q^{2/3}}{0.6^{2/3}+\ln(1+q^{1/3})}$. For a system with masses $M_{\rm WD}, M_{\rm C}$ write a function to find the separation, $a$, at which the radius of the WD is equal to $R_L$.

In [ ]:
function Roche_Lobe_contact
    #your code here
end

Now, compare your results to mine by calling `Integrate.initial_circular_orbit(M_WD; M_NS=M_NS)`. This will output a full set of initial conditions, so make sure that you are comparing the correct quantity between your function and mine.

In [ ]:
#compare your function and mine here in any way you see fit.

# Integration

We have now successfully developed the tools that we need to integrate our system! Now we will learn how to use `DifferentialEquations.jl` to solve the differential equation that we need. Let's start out with a classic example--the pendulum.

In [ ]:
function pendulum_acceleration!(du,u,p,t)
    #u=[ϕ,dϕ]
    #p=[g,L] optional parameter that determines "coupling constant" ω
    #Energy is conserved => equation is independent of time
    
    ϕ=u[1]
    dϕ=u[2]
    
    ddϕ=-p[1]/p[2] * sin(ϕ)#pendulum equation of motion
    
    du[1]=dϕ#mutating du according to the pendulum EoM
    du[2]=ddϕ 
end



`DifferentialEquations.jl` makes it very easy from here to solve the system over a set timespan:

In [ ]:
function solve_pendulum(u0,p,tspan)
    #u0: Initial conditions
    #p: system parameters
    #tspan: time over which we integrate
    
    prob = ODEProblem(pendulum_acceleration!, u0, tspan, p)
    sol = solve(prob)

    return(sol)
end

For a solution, `sol` we can call `sol(t)` to retrieve the interpolated solution at time `t`$\in$`tspan`: 

In [ ]:
t_sol=solve_pendulum([0,0.5],[1,1],(0,20))

times=range(0,stop=20,length=1001)

sol_interpolated=t_sol(times)

ϕ  = sol_interpolated[1,:]
dϕ = sol_interpolated[2,:]

plot(times,ϕ,xlabel="time",ylabel="ϕ",label="(1,1)")

In [ ]:
#Try solving the equation for a variety of different values for u0 and p
#Do all solutions have the same periodic behavior?
#Are g and L degenerate in any way?
#What does this physically represent?

Now, I am going to describe a different system where the length of the arm is allowed to change periodically with time. I would like you to try to integrate this system and describe its behavior. Does this system remind you of anything growing up?

In [ ]:
function swingy_problem!(du,u,p,t)
    #u=[ϕ,dϕ]
    #p=[g,L,L0,ω0,Φ] system parameters
    #What do each of these these physically represent?
    #Energy is conserved => equation is independent of time
    
    ϕ=u[1]
    dϕ=u[2]

    Lt=L + L0 * sin(t*ω0+Φ)
    
    ddϕ=-p[1]/Lt * sin(ϕ)#pendulum equation of motion
    
    du[1]=dϕ#mutating du according to the pendulum EoM
    du[2]=ddϕ 
end



In [ ]:
#Integrate `swingy_problem` and plot the results
#Try varying system parameters one at a time to see what they do.
#Are there any system parameters which make your integrator particularly unhappy?

Challenge: Now I would like you to write your own differential equation describing a planet orbiting a star. The equation of motion here is:$$\mathbf{r}=-\frac{GM}{r^2}\hat{\mathbf{r}}$$If you would like to include $G$ and $M$ as parameters, you are welcome to do so, but it is also okay for now just to use $G=1$ and work in units of $M$ (sidenote: this is called "Geometrized units", and they are often used in numerical relativity). Try following the pattern for solving the problem that I laid out above. 

*Hint: Your `u` array should lave a length of 6.*

In [ ]:
function one_body_motion!(du,u,p,t)
    #your code here
end

In [ ]:
#Now try integrating your equation.
#Are there any distinct regimes in which the system behaves fundamentally differently?

In principle, this is all that we have to do to determine the dynamics of our WD-CO system--plug our `Math.acceleration_even!` function into `ODEProblem` and solve. In practice, it is more difficult for several reasons. First, we do not know how long these integrations will last, so we cannot just set bounds as we do in the above problems. Second, there are times when the system evolves very slowly, and other times when the system evolves very quickly--we treat these by integrating them separately.

To solve these problems, I have written a variety of integration tools spread across `Integrate.jl`, `Integrate_r.jl`, and `Pipeline.jl` to be used under different scenarios. We will go through an integration below to try to work out how to use some of these tools. I have also tried to document them with their specific use cases in the source files, so if you have questions, I would recommend taking a look at those.

In [ ]:
#Starting out, we must determine our initial conditions
#here, we will integrate a high mass (but not extreme) WD with a typical NS 

initial_conditions=Integrate.initial_circular_orbit(1.0; M_NS=1.35)



Now, we will use `Integrate_r.integrate_r_sneaky` to integrate the inspiral phase of the system. This function integrates the system in the r-basis--we will decrease the separation by constant steps of 0.1 km.

We also integrate not the system itself, but the change in the initial conditions (thus the 'sneaky' part). The reason that we do this is that at first the system changes very slowly--so slowly that the change can be lost over one time step due to finite floating point precision--and if this happens, then the change is 0. So even if we integrate over a million time steps, $10^6\cdot0=0$.

In [ ]:
initial_conditions

In [ ]:
initial_conditions_w_time=zeros(7)#We integrate t with respect to r here, so need an extra variable
initial_conditions_w_time[1:6]=initial_conditions

phase_1_output=Integrate_r.integrate_r_sneaky(initial_conditions_w_time,-1) #The -1 tells the function that we are decreasing in separation
#This function is going to print a lot of things to let me know that it is working.
#See if you can figure out what they all mean!
#Again, you can ignore the warning that it flashes up.
#This integration may take a couple of minutes to complete

In [ ]:
phase_1_output

In a later step, we will process these data and label them to be more easily understandable. However, it might be a good exercise to stop here and try to tell what the last line means physically.

In [ ]:
#And remember that since this is a difference, to get physically meaningful terms, we need to add the initial condition:
#Note that in this particular stage, it is numerically unstable to evolve white dwarf radius dynamically.
#We treat this explicitly in our source code (you can take a look at how if so inclined),
#but the key is that we don't believe the radii that I put out 


phase_1_output_total=phase_1_output.+initial_conditions_w_time'


Next, we evolve the turnaround (between inspiral and outspiral phases) in the time basis. (Why can't we do this in the separation basis like the previous step?)

In [ ]:
new_base=phase_1_output_total[end-5,:]#Starting five steps from the end of the last integration
T=phase_1_output_total[end,end]-phase_1_output_total[end-5,end] #Characteristic turnaround time 

phase_2_output=Pipeline.turn_around_dtheta_dt(new_base[1:6],T)



In [ ]:
phase_2_output_total=phase_2_output.+new_base'

We then try to evolve the "outspiral" (where $\dot a>0$) in the separation basis. If this fails, it likely means that we are still too close to $\dot a=0$, so we integrate in the $t$-basis until we are finally able to integrate in $a$:  

In [ ]:
newer_base=phase_2_output_total[end,:]

phase_3_output_total=Pipeline.turn_around_and_out(newer_base,T) 
#This function already adds the basis in for us.

What we would like the function to do for us it to continue to integrate until $M_{\rm WD}<0.05\,M_\odot$, but it turns out that this is more difficult for Isotropic Re-emission scenarios (as the code is currently set for) than it is for Jeans mode. Instead, I will just ask it to integrate until the separation is wider than the white dwarf would be at $0.05\,M_\odot$:


In [ ]:
newest_base=phase_3_output_total[end,:]

phase_4_output_total=Pipeline.turn_around_and_out_terminate(newest_base,T)#This might take a few minutes to run



In [ ]:
phase_3_output_total

Now we will concatenate these outputs to form a full timeseries solution:

In [ ]:
println()

full_out=vcat(phase_1_output_total[1:end-6,:],phase_2_output_total)
full_out=vcat(full_out[1:end-1,:],phase_3_output_total)
full_out=vcat(full_out[1:end-1,:],phase_4_output_total)

In principle, `Pipeline.integrate_r_full` or `Pipeline.integrate_r_full2` should do all of this for you. In practice, this pipeline is somewhat brittle (especially for isotropic re-emission sources), and it is good to know what building blocks to put together to get the integration that you need out (especially because you do not need the full integration every time).

We can also calculate the gravitational wave frequency and quadrupole second derivative:

In [ ]:
waveform_out=Pipeline.package_quadrupole_frequency(full_out)
#What do you think each of these columns means?

In [ ]:
df_waveform=DataFrame(waveform_out, ["t", "f", "ddI_p", "ddI_c", "a", "M_WD", "M_NS", "Phase"])
#look at the source code for Pipeline.package_quadrupole_frequency.
#How do we interpret ddI_P vs. ddI_c?

Now, you can call `df_waveform` columns by name to get them out. For instance

In [ ]:
plot(df_waveform[!,"t"],df_waveform[!,"f"],xlabel="t [s]",ylabel="f [Hz]")

Now that you have seen how an integration is done, I would like you to try to analyze this signal and run a few integrations yourself:
1. Following my paper draft, try to recreate the style of plot that I made in Fig. 1/7 for the above run (e.g., $a,\,f,\,M_{\rm WD},$ and $h$ vs. $t$). The style of the plot is not important (i.e., don't spend a lot of time trying to figure out how to make it multipanel or splitting the inspiral vs. outspiral axes--basic, separate plots are fine here). Note units in calculating strain, $h$, and assume a distance of $D_L=1\,\mathrm{ Mpc}$ and face-on geometry.
2. Use the stationary phase approximation discussed in my paper draft to recreate my Figure 2.
3. Try exploring some of the other functions in my `Integrate.jl`, `Integrate_r.jl`, and `Pipeline.jl` files. Choose a few, and try to figure out how/why to use them.
4. Run integrations for varying sets of initial conditions. In particular, try varying the compact partner mass by a couple of orders of magnitude. Do any of these systems undergo a common envelope merger as described in my section 4.2.2 or Appendix A? Compare your results to my figure 8.
5. Challenge: Run an analogous integration and make plots for angular momentum loss under the assumption of Jeans Mode (right now we are using Isotropic Re-emission). *Hint: You will need to change the source code because I have hardcoded this assumption and not yet gotten around to allowing the user to change the optional parameters from their default values.* 

In [ ]:
#happy coding!